In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)

True

In [3]:
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [7]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [9]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

In [50]:
system_input=""""
You will be provided lots of links and data of a webpage,
you will Analyze it and pick the most relevent ones to make a company broucher page.
give the output in this json format:
{
"pages":[
    {"id":"company about", "link":"https://..../about","summary:"summary of that page"},
            {"id":"career page", "link":"https://..../career","summary:"summary of that page"}}

]
}
in link give the actual output.
"""


In [54]:
def user_input_format(x):
    user_input=f""" Here is the list of {x}
    Please decide which of these are relevant web links for a brochure about the company. 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

    """
    url=fetch_website_links(x)
    user_input+= "\n".join(url)
    return user_input


In [55]:
print(user_input_format("https://edwarddonner.com"))

 Here is the list of https://edwarddonner.com
    Please decide which of these are relevant web links for a brochure about the company. 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

    https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2025/11/11/ai-live-event/
ht

In [ ]:
def select_relevent_links(url):
    response=openai.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":system_input },{"role":"user","content":user_input_format(url)}],
        response_format={"type":"json_object"}
    )
    result=json.loads(response.choices[0].message.content)
    return result

In [60]:
print(type(select_relevent_links("https://edwarddonner.com")))

<class 'dict'>


In [62]:
select_relevent_links("https://edwarddonner.com")

{'pages': [{'id': 'home',
   'link': 'https://edwarddonner.com/',
   'summary': "Overview of Edward Donner's work, focusing on AI-enabled solutions and Nebula projects."},
  {'id': 'about-nebula',
   'link': 'https://edwarddonner.com/about-me-and-about-nebula/',
   'summary': 'Biographical page describing Edward Donner and the Nebula initiative, its mission and expertise.'},
  {'id': 'curriculum',
   'link': 'https://edwarddonner.com/curriculum/',
   'summary': 'Professional background and experience (CV) detailing education, projects, and roles.'},
  {'id': 'proficient',
   'link': 'https://edwarddonner.com/proficient/',
   'summary': 'Summary of skills and areas of proficiency relevant to AI, LLMs, and product development.'},
  {'id': 'connect-four',
   'link': 'https://edwarddonner.com/connect-four/',
   'summary': 'Connect Four project page showcasing problem solving, design, and interactive capabilities.'},
  {'id': 'outsmart',
   'link': 'https://edwarddonner.com/outsmart/',
   '

In [59]:
# links = [page["link"] for page in select_relevent_links()["pages"]]
# print(links)

In [ ]:
def fetch_page_with_link(link):
    contents = fetch_website_contents(link)
    links=select_relevent_links(link)
    result=f"## Landing page:\n\n {contents}\n ## Relevent Links:\n"
    for item in links["pages"]:
        result+=f"\n\n#This is the Title of the Page:{item["id"]}\n"
        result+=fetch_website_contents(item["link"])
    return result

In [72]:
print(fetch_page_with_link("https://edwarddonner.com"))

## Landing page:

 Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 

In [73]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [74]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_with_link(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [75]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing page:\n\n Hugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/personaplex-7b-v1\nUpdated\n6 days ago\n•\n43.9k\n•\n1.31k\nmoonshotai/Kimi-K2.5\nUpdated\nabout 11 hours ago\n•\n11k\n•\n792\nmicrosoft/VibeVoice-ASR\nUpdated\n1 day ago\n•\n76.7k\n•\n685\nQwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\nUpdated\n5 days ago\n•\n139k\n•\n669\nTongyi-MAI/Z-Image\nUpdated\nabout 21 hours ago\n•\n506\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n904\nQ

In [84]:
def broucher_create(name,url):
    response=openai.chat.completions.create(model="gpt-4.1-mini",messages=[{"role":"system","content":brochure_system_prompt},
                                            {"role":"user","content":get_brochure_user_prompt(name,url)}]
                                            )
    result=response.choices[0].message.content
    display(Markdown(result))

In [86]:
broucher_create("Discord", "https://discord.com/")

# Discord - The Ultimate Group Chat Experience

---

## About Discord

Discord is a free voice, video, and text chat platform designed to bring people together. Originally built for gamers to "play and chill with friends," it has evolved into a thriving global community space where users can create customized "servers" to talk, play, and hang out. Whether you want to game, share photos, watch shows, or simply chat, Discord offers seamless, high-quality, low-latency streaming to make you feel like you’re in the same room.

---

## What Makes Discord Unique?

- **All-in-One Communication:** Voice, video, text chat, and screen sharing with friends or communities.
- **Customizable Spaces:** Personalize your server with custom emojis, stickers, soundboard effects, avatars, statuses, and profiles that express your personality.
- **Worldwide Community Building:** Create or join communities on any topic—from gaming, hobbies, education, or business.
- **Nitro Subscription:** Unlock premium features including enhanced emojis, bigger upload limits, improved streaming quality, and more.

---

## Commitment to Safety and Wellbeing

Discord is deeply committed to maintaining a safe and welcoming community environment. It offers extensive safety resources through:

- **Family Center & Safety Library:** Tools and guidelines to support parents and teens.
- **Teen Charter & Wellbeing Hub:** Empower young users and ensure their online safety.
- **Transparency and Policy Hubs:** Clear policies on privacy, safety, and moderation.
- **Quests:** Educational resources to foster responsible and enjoyable use of the platform.

---

## Customers and Community

Discord caters to millions of users worldwide including:

- Gamers seeking seamless communication.
- Global communities of all interests and sizes.
- Developers using Discord Social SDK and APIs to create apps and activities on the platform.
- Creators and streamers who engage followers with live high-quality streams.

---

## Developer & Engineering Opportunities

Discord supports developers with comprehensive documentation, tools, and a developer newsletter to encourage innovative apps and extensions. The platform nurtures a vibrant ecosystem where engineers and creators can grow together.

---

## Careers at Discord

Discord offers exciting career paths in:

- Engineering & Development
- Product & Feature Design
- Policy & Safety
- Marketing & Community Management

As a company headquartered in San Francisco, CA, with a strong international presence (e.g., Netherlands office), Discord values innovation, inclusivity, and creating experiences that connect people globally.

---

## Company Information

**Discord Inc. Headquarters**  
444 De Haro Street, Suite 200  
San Francisco, CA 94107  
United States  

**Contact:** support@discord.com | Phone: 888-594-0085

**Discord Netherlands B.V.**  
Schiphol Boulevard 195  
1118BG Schiphol, Netherlands  
Phone: +31 20 809 0400  

Authorized Representative: Clint Smith, CLO  
Registered in Delaware, USA | FEIN: 45-4908598  

---

## Get Started with Discord Today!

- Download Discord for Windows or Mac, or use the browser version.  
- Join millions who use Discord as their daily communication hub.  
- Customize your space, make new friends, and explore communities worldwide.

**Website:** [discord.com](https://discord.com)  
**Support:** https://support.discord.com

---

_Discover the chat platform that’s all fun and games—join Discord and be part of something bigger._

In [85]:
broucher_create("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is a leading AI community and collaboration platform dedicated to building the future of machine learning. It serves as a vibrant ecosystem where ML engineers, scientists, developers, and enthusiasts collaborate openly on a vast range of machine learning models, datasets, and applications. Their platform hosts over 2 million models and 500,000+ datasets, empowering millions to create, discover, and share AI innovations.

---

## What They Offer  
- **Hugging Face Hub**: Central repository and collaboration space to host, share, and explore models, datasets, and machine learning applications.  
- **Models**: Wide-ranging pre-trained models from major contributors like NVIDIA, Microsoft, and Alibaba, across modalities including text, image, video, audio, and 3D data.  
- **Datasets**: Access to a vast and growing library of datasets to fuel ML experiments and solutions.  
- **Spaces**: Interactive AI applications and demos powered by community and enterprise contributors, enabling hands-on exploration of ML capabilities.  
- **Open Source Stack**: Tools and infrastructure designed for faster development and deployment of ML projects.  
- **Community**: A thriving global community providing collaboration, learning, and open exchange of ideas to drive ethical and open AI forward.

---

## Company Culture  
Hugging Face fosters a culture of openness, collaboration, and ethical innovation. It emphasizes empowering users and contributors to build and share their work openly, creating a supportive environment aimed at the collective growth of AI and machine learning. The company champions transparency and inclusivity within the AI community, encouraging participation across skill levels to build a trustworthy AI future.

---

## Customers & Community  
Their customer base includes top-tier technology companies such as NVIDIA, Microsoft, Alibaba, and other leaders in AI development who contribute and leverage Hugging Face's platform. Additionally, thousands of smaller companies, academic institutions, and independent developers actively engage with the platform.

The community is fast-growing and diverse, playing a crucial role in evolving AI technologies, sharing best practices, and driving the open source ML movement worldwide.

---

## Careers at Hugging Face  
Hugging Face offers career opportunities for individuals passionate about AI and the open source ethos. Working at Hugging Face means being part of a forward-thinking team focused on cutting-edge research, community-building, and building tools that accelerate machine learning adoption. Job roles typically span machine learning engineering, research, software development, product management, and community engagement.

The company culture values creativity, collaboration, continuous learning, and making an impact by advancing open and ethical AI technologies.

---

## Contact & Explore  
- Visit the platform: [huggingface.co](https://huggingface.co)  
- Explore over 2M ML models and 500k+ datasets  
- Join the community to collaborate, share, and innovate  
- Discover AI applications with interactive Spaces demos

---

## Brand Identity  
Hugging Face’s brand is vibrant and approachable, combining a dynamic yellow (#FFD21E) and orange (#FF9D00) palette with modern typography to reflect its energetic commitment to open AI collaboration. Their logo and assets symbolize the friendly and inclusive spirit of the community.

---

### Join Hugging Face to build the future of AI — together.